# 02 - Modelo Markowitz

Valida el optimizador Markowitz sin tasa libre de riesgo. El objetivo es resolver media-varianza `mu*w - 0.5*gamma*w?w` con retornos semanales totales, shrinkage de covarianza y l?mites por perfil. El Sharpe se reporta como `retorno_anual / volatilidad_anual`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from markowitz_pipeline import (
    DEFAULT_MAX_WEIGHT,
    build_profile_subuniverses,
    load_prepared_outputs,
    optimize_markowitz,
    equal_weight_metrics,
    _clean_returns_for_tickers,
    resolve_paths,
)

paths = resolve_paths(Path(r"X:/capstone/Modelo_finanzas"))
universe_f5, daily_returns, weekly_returns, mu, sigma = load_prepared_outputs(paths)
print(len(universe_f5), daily_returns.shape, mu.shape, sigma.shape)

In [ ]:
subuniverses = build_profile_subuniverses(universe_f5)
profile = "Neutro"
tickers = subuniverses[subuniverses["profile"] == profile]["ticker"].tolist()
returns = _clean_returns_for_tickers(daily_returns, tickers)

weights, metrics = optimize_markowitz(returns, max_weight=DEFAULT_MAX_WEIGHT)
eq_weights, eq_metrics = equal_weight_metrics(returns)

print(profile)
print(metrics)
print(eq_metrics)

In [ ]:
comparison = pd.DataFrame([
    {"portfolio": "Markowitz", **{k: metrics[k] for k in ["annual_return", "annual_volatility", "sharpe"]}},
    {"portfolio": "Equiponderado", **{k: eq_metrics[k] for k in ["annual_return", "annual_volatility", "sharpe"]}},
])
comparison["delta_sharpe_vs_equal_weight"] = comparison["sharpe"] - eq_metrics["sharpe"]
comparison

In [ ]:
top_weights = (
    weights.sort_values(ascending=False)
    .head(15)
    .rename("weight")
    .reset_index()
    .rename(columns={"index": "ticker"})
    .merge(universe_f5[["ticker", "sector", "market_cap", "short_name"]], on="ticker", how="left")
)
top_weights

In [ ]:
assert np.isclose(weights.sum(), 1.0, atol=1e-6)
assert weights.ge(-1e-9).all()
assert weights.le(PROFILE_CONFIGS[profile].max_weight + 1e-6).all()
assert np.isfinite(metrics["sharpe"])
print("Validaciones Markowitz OK")